# Tool calling against FHIR data

Exercise notebook for the lesson **[Tool calling against FHIR data](https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling)** in the *Agentic AI* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** FHIR gives agents a standard way to read health records: resources such as Observation and MedicationRequest, searched through web APIs. Safe agent tools are narrow (one question each), read-only by default, scoped to the patient the clinician has open rather than chosen by the model, return values with their source resource and date, and log every call. Authorisation frameworks such as SMART on FHIR limit what an app can see in the first place.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### A synthetic FHIR bundle

Two synthetic patients, three observations and two medication requests.

In [ ]:
# A small synthetic FHIR R4 bundle (made-up patients, no real data)
bundle = {"resourceType": "Bundle", "entry": [{"resource": r} for r in [
    {"resourceType": "Patient", "id": "p1", "birthDate": "1958-04-02", "gender": "female"},
    {"resourceType": "Patient", "id": "p2", "birthDate": "1971-09-15", "gender": "male"},
    {"resourceType": "Observation", "id": "o1", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-01-10", "valueQuantity": {"value": 8.2, "unit": "%"}},
    {"resourceType": "Observation", "id": "o2", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "4548-4", "display": "Hemoglobin A1c"}]},
     "effectiveDateTime": "2026-07-02", "valueQuantity": {"value": 7.4, "unit": "%"}},
    {"resourceType": "Observation", "id": "o3", "subject": {"reference": "Patient/p1"},
     "code": {"coding": [{"system": "http://loinc.org", "code": "98979-8", "display": "eGFR (CKD-EPI 2021)"}]},
     "effectiveDateTime": "2026-07-02", "valueQuantity": {"value": 41, "unit": "mL/min/1.73m2"}},
    {"resourceType": "MedicationRequest", "id": "m1", "status": "active", "subject": {"reference": "Patient/p1"},
     "medicationCodeableConcept": {"text": "Metformin 500 mg tablets"}, "authoredOn": "2024-05-01"},
    {"resourceType": "MedicationRequest", "id": "m2", "status": "stopped", "subject": {"reference": "Patient/p1"},
     "medicationCodeableConcept": {"text": "Gliclazide 80 mg tablets"}, "authoredOn": "2023-02-11"},
]]}
resources = [e["resource"] for e in bundle["entry"]]
print({t: sum(r["resourceType"] == t for r in resources)
       for t in ["Patient", "Observation", "MedicationRequest"]})

# {'Patient': 2, 'Observation': 3, 'MedicationRequest': 2}

### Narrow, read-only, patient-scoped tools

Continuing from the bundle block: patient-scoped tools with provenance and an audit log.

In [ ]:
class PatientTools:
    """Read-only tools scoped to ONE patient, chosen by the clinician, not by the model."""
    def __init__(self, resources, patient_id):
        self.patient_ref = f"Patient/{patient_id}"
        self.resources = resources
        self.audit = []

    def _mine(self, resource_type):
        return [r for r in self.resources if r["resourceType"] == resource_type
                and r.get("subject", {}).get("reference") == self.patient_ref]

    def latest_observation(self, loinc_code):
        self.audit.append(("latest_observation", loinc_code))
        matches = [o for o in self._mine("Observation")
                   if any(c["code"] == loinc_code for c in o["code"]["coding"])]
        if not matches:
            return {"result": None, "note": f"no observation with LOINC {loinc_code}"}
        o = max(matches, key=lambda o: o["effectiveDateTime"])
        return {"value": o["valueQuantity"]["value"], "unit": o["valueQuantity"]["unit"],
                "date": o["effectiveDateTime"], "source": f"Observation/{o['id']}"}

    def active_medications(self):
        self.audit.append(("active_medications", None))
        return [{"medication": m["medicationCodeableConcept"]["text"],
                 "source": f"MedicationRequest/{m['id']}"}
                for m in self._mine("MedicationRequest") if m["status"] == "active"]

tools = PatientTools(resources, patient_id="p1")
print(tools.latest_observation("4548-4"))
print(tools.latest_observation("98979-8"))
print(tools.active_medications())
print(tools.latest_observation("2093-3"))
print("audit log:", tools.audit)

# {'value': 7.4, 'unit': '%', 'date': '2026-07-02', 'source': 'Observation/o2'}
# {'value': 41, 'unit': 'mL/min/1.73m2', 'date': '2026-07-02', 'source': 'Observation/o3'}
# [{'medication': 'Metformin 500 mg tablets', 'source': 'MedicationRequest/m1'}]
# {'result': None, 'note': 'no observation with LOINC 2093-3'}
# audit log: [('latest_observation', '4548-4'), ('latest_observation', '98979-8'), ('active_medications', None), ('latest_observation', '2093-3')]

## Exercise

**Goal:** Build and test patient-scoped FHIR tools over synthetic data, then connect them to the agent loop.

**Data:** Synthetic FHIR bundle in the notebook; optional Synthea FHIR R4 bundles (Generated in the notebook; Synthea is Apache 2.0)

### Step 1

Reproduce the tools and add one for active conditions (Condition resources) and one for allergies.

In [ ]:
# Your code here


### Step 2

Add a 'stale' flag when the latest observation is more than 12 months old.

In [ ]:
# Your code here


### Step 3

Try to read patient p2's data through the p1 tools and confirm it is impossible.

In [ ]:
# Your code here


### Step 4

Plug the tools into the agent loop from the previous lesson and have the scripted model produce a summary with source references.

In [ ]:
# Your code here


### Step 5

Optional: load a Synthea FHIR bundle and run the same tools on it.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling](https://ramkumarmd.com/medical-ai/agentic-ai/fhir-tool-calling). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.